### Hier werden die Daten gereinigt und dann einerseits der ganze gereinigte Datensatz abgespeichert und andererseits eine Excel-Datei mit rund 1000 Fällen für die manuelle Codierung erstellt. Abschließend wird die Interrater Reliabilität berechnet.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import os
import pandas as pd
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt
import re
import emoji
from sklearn.model_selection import train_test_split
from PIL import Image
import pytesseract
from datetime import time
from sklearn.metrics import cohen_kappa_score


os.chdir(r"C:\Users\hundh\Desktop\Python\Daten")
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("data_btw_final_ocr.csv")
df.shape[0]

In [ ]:
# Bildbeschreibung und OCR-Text werden zusammengelegt (und mit " --- " getrennt).
df["body"] = df["body"].astype(str)
df["ocr_text"] = df["ocr_text"].astype(str)

df["body"] = df[["ocr_text","body"]].agg(' ---------- '.join, axis=1)

In [ ]:
# Nur die im 20. deutschen Bundestag mit mehr als einem Abgeordneten/einer Abgeordneten vertretenen Parteien werden untersucht
df = df[(df["party"]=='CDU') | (df["party"]=='CSU') | (df["party"]=='SPD') | (df["party"]=='AfD') | (df["party"]=='FDP') | (df["party"]=='GRÜNE') | (df["party"]=='Die Linke') | (df["party"]=='BSW') | (df["party"]=='GRÜNE/B 90')]
df["party"] = df["party"].replace({'GRÜNE/B 90':"GRÜNE"})

df = df[df["timestamp"] > "2024-11-06T00:00:00Z"] #Start des Erhebungszeitraums
df = df[df["timestamp"] <= "2025-02-23T18:00:00Z"] # Am Wahltag um 18:00 Uhr war Ende der Wahlabgabe

df.shape[0] # Anzahl nach Entfernung der Posts von Kandidierenden anderer Parteien und der Posts außerhalb des Untersuchungszeitraums

In [ ]:
# Zeitvariable
df[["date","time"]] = df["timestamp"].str.split("T", expand=True)  #Bei Zeit hinter Variable immer ein Z. Wenn nötig löschen
df['date'] = pd.to_datetime(df['date'], errors="coerce")

# Aus Zeitvariable wird eine Variable mit 5 Zeitabschnitten gebildet. Diese wird für die Statifizierung verwendet
periods_5 = [(df["date"].lt("2024-11-28")),  # 6.11 - 28.11 (draußen)
         (df["date"].ge("2024-11-28") & df["date"].lt("2024-12-20")), # 28.11 - 20.12 (draußen)
         (df["date"].ge("2024-12-20") & df["date"].lt("2025-01-11")), # 20.12 - 11.01 (draußen)
         (df["date"].ge("2025-01-11") & df["date"].lt("2025-02-02")), # 11.01. - 02.02 (draußen)
         (df["date"].ge("2025-02-02"))] # 02.02. - 23.02 (drinnen)

choices_5 = ["period_1", "period_2", "period_3", "period_4", "period_5"]

df["period"] = np.select(periods_5, choices_5, default="unknown")

In [ ]:
#Posts von Kandidierenden, bei denen diese nur Co-Autoren sind, werden gelöscht
df = df[df["author"] == df["acc_name"]]
df.shape[0]

In [ ]:
print(df[df["acc_type"] != "candidate"].groupby("author_fullname", as_index=False).size()) #111 Parteiaccounts
df = df[df["acc_type"] == "candidate"] #Posts von Parteiaccounts werden gelöscht
print(df[df["sex"] == "d"].groupby("author_fullname", as_index=False).size()) # Eine nicht binäre Person mit 8 Posts
df = df[df["sex"] != "d"] #non-binäre Person wird gelöscht.

In [ ]:
# Das ist die finale Anzahl der Fälle, die eine Issue-Kodierung erhalten
df.shape[0]

In [ ]:
# Es wird eine Variable erstellt um bei jedem Kandidierenden unterscheiden zu können, ob sein Wahlkreis (oder sein Listenplatz) in Ost- oder Westdeutschland liegt

conditions = [(df["state"].eq("NI") |df["state"].eq("HB") |df["state"].eq("BY") |df["state"].eq("BW") |df["state"].eq("NW") |
               df["state"].eq("SL") |df["state"].eq("HE") |df["state"].eq("SH") |df["state"].eq("HH") |df["state"].eq("RP") |
               df["state"].eq("BE")),
               (df["state"].eq("SN") |df["state"].eq("ST") |df["state"].eq("BB") |df["state"].eq("MV") |df["state"].eq("TH"))]

choices = ["West","Ost"]

df["Ost_West"] = np.select(conditions, choices, default="unknown")

In [ ]:
# Hier werden Zeichen aus Posts entfernt, die nur durch das Kopieren der Posts von Instagram raus entstehen (\n und \r). Emojis werden auch entfernt. Für die bessere Lesbarkeit werden große Lücken verkleinert

df["body"] = df["body"].astype(str)
df["body"] = df["body"].map(lambda x: str(x.replace("\n"," "))) 
df["body"] = df["body"].map(lambda x: str(x.replace("\r"," "))) 
df["body"] = df["body"].map(lambda x: emoji.replace_emoji(x,replace=""))
df["body"] = df["body"].map(lambda x: str(x.replace("   "," "))) 
df["body"] = df["body"].map(lambda x: str(x.replace("  "," ")))

df.rename(columns={"body":"text"}, inplace=True)

In [ ]:
# Der gesäuberte Datensatz wird abgespeichert
df.to_csv("kandis_cleaned.csv")

### Als nächstes wird ein Teil der Daten für die manuelle Kodierung in einer Excel-Tabelle abgespeichert

In [ ]:
df = df.reset_index()
df = df.rename(columns={"index":"nummer"})

In [ ]:
# Um sicher zu gehen, dass wichtige Faktoren gelich häufig in der Excel-Tabelle vorkommen, wird stratifiziert
# Die Anzahl einzelner Gruppen aus den Kombinationen ist sehr gering, deshalb kann eine noch strengere Stratifizierung nicht durchgeführt werden
df.groupby(["sex","party","Ost_West","period"], as_index=False).size().sort_values(by="size") 

In [ ]:
# Stratifizierung nach Partei, Geschlecht, Ost_West und Wahlkampfperiode.
train, test = train_test_split(df, test_size=0.987, stratify=df[["party","sex", "Ost_West","period"]], random_state=111)
train.shape[0]

In [ ]:
# Die 1086 ausgewählten Fälle werden in einer Excel-Tabelle manuell codiert
df_export = train[["text"]]
file_name = "Excel_eigencodierung/Excel_eigencodierung.xlsx"
df_export.to_excel(file_name)

### Im nächsten Schritt wird die manuelle Kodierung evaluiert

In [ ]:
coder1 = pd.read_excel("Excel_eigencodierung/Excel_eigencodierung_Lucas.xlsx")
coder2 = pd.read_excel("Excel_eigencodierung/Excel_eigencodierung_Phillip.xlsx")

coder1.dropna(subset="text", inplace=True)
coder2.dropna(subset="text", inplace=True)

Da keine klaren "best-practice" Maße für die inter-rater Reliabilität identifizierbar waren, haben wir für jede Kategorie Cohen`s Kappa berechnet und dann daraus den Mittelwert als Maß für die durchschnittliche inter-rater Reliabilität weiterverwendet

In [ ]:
coder1_qual = coder1.copy(deep=True)
coder2_qual = coder2.copy(deep=True)

del coder1["nummer"]
del coder1["text"]
del coder2["nummer"]
del coder2["text"]

coder1_D0 = coder1["D_0"]
coder1_D1 = coder1["D_1"]
coder1_D2 = coder1["D_2"]
coder1_D3 = coder1["D_3"]
coder1_D4 = coder1["D_4"]
coder1_D5 = coder1["D_5"]
coder1_D6 = coder1["D_6"]
coder1_D7 = coder1["D_7"]

coder2_D0 = coder2["D_0"]
coder2_D1 = coder2["D_1"]
coder2_D2 = coder2["D_2"]
coder2_D3 = coder2["D_3"]
coder2_D4 = coder2["D_4"]
coder2_D5 = coder2["D_5"]
coder2_D6 = coder2["D_6"]
coder2_D7 = coder2["D_7"]


In [ ]:
print(cohen_kappa_score(coder1_D0, coder2_D0))
print(cohen_kappa_score(coder1_D1, coder2_D1))
print(cohen_kappa_score(coder1_D2, coder2_D2))
print(cohen_kappa_score(coder1_D3, coder2_D3))
print(cohen_kappa_score(coder1_D4, coder2_D4))
print(cohen_kappa_score(coder1_D5, coder2_D5))
print(cohen_kappa_score(coder1_D6, coder2_D6))
print(cohen_kappa_score(coder1_D7, coder2_D7))

kappa0 = cohen_kappa_score(coder1_D0, coder2_D0)
kappa1 = cohen_kappa_score(coder1_D1, coder2_D1)
kappa2 = cohen_kappa_score(coder1_D2, coder2_D2)
kappa3 = cohen_kappa_score(coder1_D3, coder2_D3)
kappa4 = cohen_kappa_score(coder1_D4, coder2_D4)
kappa5 = cohen_kappa_score(coder1_D5, coder2_D5)
kappa6 = cohen_kappa_score(coder1_D6, coder2_D6)
kappa7 = cohen_kappa_score(coder1_D7, coder2_D7)

kappa = [kappa0, kappa1, kappa2, kappa3, kappa4, kappa5, kappa6, kappa7]

mean = sum(kappa) / len(kappa)
print(f"Kohens kappa Mittelwert:{mean}")

Im weiteren Verlauf werden die Fälle herausgesucht, bei denen die beiden Coder nicht übereinstimmten. Bei diesen Fällen haben wir uns auf eine gemeinsame Kodierung geeinigt.

In [ ]:
disagreement = coder1_qual != coder2_qual
mask = disagreement.any(axis=1)

disagreements_df = pd.concat(
    {"coder1": coder1[mask], "coder2": coder2[mask]},
    axis=1
)

disagreements_df = disagreements_df.reset_index()

coder1_qual = coder1_qual.reset_index()

disagreements_df.rename(columns={"":"row"}, inplace=True)

disagreements_df.columns = disagreements_df.columns.droplevel()

disagreements_df["row_excel"] = disagreements_df["row"] + 2 # +2 weil der Index hier bei 0 beginnt und in Excel (quasi) bei 2

disagreements_df["row_excel"]

Bei den ungleich codierten Fällen wurde sich gemeinsam auf eine Codierung geeinigt. Die gemeinsame Codierung liegt in der Excel-Datei "Excel_eigencodierung_final" vor